# 🔧 Smoke Test: `visualize_interactive.py` + ipympl widget backend

这不是正式课程 notebook（00-05），而是一个**开发期验证脚本**，用来确认：

1. `%matplotlib widget`（由 `ipympl` 提供）后端在你的环境里能正常工作；
2. `visualize_interactive.py` 里的 7 个函数分别都能正确画图；
3. 滑块拖动时图像是**原地平滑更新**（不是每次都弹出一张新的静态图）。

**运行前**：确保已 `pip install -r requirements.txt`（包含 `ipywidgets`、`jupyterlab`、`ipympl`），并且从 Jupyter 里打开本文件（`jupyter lab` 或 `jupyter notebook`，不要用 VS Code 的 notebook 预览，某些版本对 `ipympl` 支持不完整）。

**如何判断“工作平滑”**：拖动下面任意一个滑块时，图像应该在 <1 秒内跟着更新，且**不会**在输出区域往下新增一张张图片、也不会整个 cell 重新闪烁。如果看到图片不断往下堆积，说明 `%matplotlib widget` 没生效（可能退回到了 `inline` 后端），检查 `ipympl` 是否装好。

In [ ]:
%matplotlib widget

import sys
sys.path.insert(0, '..')  # so we can import repo modules from notebooks/

import torch
import matplotlib.pyplot as plt
from ipywidgets import interact, interact_manual, FloatSlider, IntSlider, Checkbox

from toy_data import get_toy_data
from models import EnergyNet
from samplers import LangevinSampler
from losses import ContrastiveDivergenceLoss
import visualize_interactive as vi

torch.manual_seed(0)
print('✓ Imports OK. If %matplotlib widget is working, the plots below will be interactive (zoom/pan icons in the toolbar).')

## Test 0 — Baseline sanity check

先画一个最简单的图，确认 widget 后端本身没问题（应该看到一个带工具栏——放大镜/平移图标——的交互式 figure，而不是纯静态 PNG）。

In [ ]:
fig0, ax0 = plt.subplots(figsize=(4, 3))
ax0.plot([1, 2, 3], [3, 1, 2])
ax0.set_title('If you see a toolbar (zoom/pan icons) above this plot, ipympl is working')
plt.show()

## Test 1 — `redraw_energy_landscape`

这是所有 widget 回调都会用到的核心工具：`ax.clear()` + 重绘 + `fig.canvas.draw_idle()`。

拖动下面的滑块（改变一个**未训练**网络的隐藏层维度），能量曲面应该**原地**跟着变化——曲面从接近平坦（小 hidden_dim）变得更“皱”（大 hidden_dim）。这正是 notebook 00 里“模型容量”教学点要用到的交互模式。

In [ ]:
data = get_toy_data('gmm', 300)

fig1, ax1 = plt.subplots(figsize=(6, 6))
plt.show()

@interact(hidden_dim=IntSlider(min=2, max=256, step=2, value=8),
          n_hidden_layers=IntSlider(min=1, max=4, step=1, value=2))
def _test_redraw(hidden_dim, n_hidden_layers):
    torch.manual_seed(0)  # fixed init so only capacity changes, not random seed
    net = EnergyNet(hidden_dim=hidden_dim, n_hidden_layers=n_hidden_layers)
    vi.redraw_energy_landscape(fig1, ax1, net, data,
                                title=f'Untrained EnergyNet (hidden_dim={hidden_dim}, layers={n_hidden_layers})')

## Test 2 — `plot_langevin_trajectory`

先跑一个很短的 CD 训练，得到一个有点结构（多峰）的能量面，再用 `return_trajectory=True` 采样一条完整轨迹，检查粒子路径是否正确叠加在等高线图上（颜色应从浅到深，末端有星号标记）。

In [ ]:
# Quick CD training just to get a non-trivial energy landscape to sample from
torch.manual_seed(0)
energy_net = EnergyNet(hidden_dim=64, n_hidden_layers=2)
sampler = LangevinSampler(energy_net, step_size=0.1)
loss_fn = ContrastiveDivergenceLoss(energy_net, sampler, k=10)
optimizer = torch.optim.Adam(energy_net.parameters(), lr=1e-3)

for epoch in range(40):
    optimizer.zero_grad()
    loss = loss_fn(data)
    loss.backward()
    optimizer.step()
print(f'quick warm-up training done, final loss={loss.item():.4f}')

# Sample a full trajectory of 8 particles for 40 steps
init_particles = torch.randn(8, 2) * 3
trajectory = sampler.sample(init_particles, n_steps=40, return_trajectory=True)
print('trajectory shape:', trajectory.shape, '  (expected: (41, 8, 2))')

fig2, ax2 = plt.subplots(figsize=(6, 6))
vi.plot_langevin_trajectory(trajectory, energy_network=energy_net, data_samples=data, ax=ax2)
plt.show()

## Test 3 — `plot_trajectory_step` (滑块单步回放)

复用上面算出的 `trajectory`。拖动滑块应该看到轨迹**逐步伸长**、当前位置的高亮圆点跟着移动——这是 notebook 01 里“手动单步回放 MCMC 链”的交互原型。

In [ ]:
fig3, ax3 = plt.subplots(figsize=(6, 6))
plt.show()

n_steps_available = trajectory.shape[0] - 1

@interact(step=IntSlider(min=0, max=n_steps_available, step=1, value=0))
def _test_step(step):
    vi.plot_trajectory_step(trajectory, step, ax3, energy_network=energy_net,
                             title=f'Langevin chain — step {step}/{n_steps_available}')
    fig3.canvas.draw_idle()

## Test 4 — `plot_score_field`

画出 `-∇E(x)` 箭头场。箭头应该大致指向数据点聚集的方向（“下坡”）。

In [ ]:
fig4, ax4 = plt.subplots(figsize=(6, 6))
vi.plot_score_field(energy_net, ax=ax4, title='Score field of the warmed-up EnergyNet')
ax4.scatter(data[:, 0], data[:, 1], s=5, alpha=0.3, c='blue')  # overlay data for reference
plt.show()

## Test 5 — `plot_noise_vs_data`

拖动滑块改变高斯噪声分布的标准差，应该看到红色噪声点云原地跟着变胖/变瘦，蓝色真实数据点不变。

In [ ]:
fig5, ax5 = plt.subplots(figsize=(6, 6))
plt.show()

@interact(noise_std=FloatSlider(min=0.1, max=4.0, step=0.1, value=1.5))
def _test_noise(noise_std):
    noise_dist = torch.distributions.MultivariateNormal(torch.zeros(2), torch.eye(2) * noise_std**2)
    noise_samples = noise_dist.sample((300,))
    ax5.clear()
    vi.plot_noise_vs_data(data, noise_samples, ax=ax5, title=f'noise_std={noise_std:.1f}')
    fig5.canvas.draw_idle()

## Test 6 — `plot_loss_curve`

用刚才 warm-up 训练顺手记录的 loss 历史画一条曲线（这里补跑一次并记录，方便单独验证这个函数）。

In [ ]:
torch.manual_seed(1)
net_for_curve = EnergyNet(hidden_dim=64)
sampler2 = LangevinSampler(net_for_curve, step_size=0.1)
loss_fn2 = ContrastiveDivergenceLoss(net_for_curve, sampler2, k=10)
optimizer2 = torch.optim.Adam(net_for_curve.parameters(), lr=1e-3)

loss_history = []
for epoch in range(60):
    optimizer2.zero_grad()
    loss = loss_fn2(data)
    loss.backward()
    optimizer2.step()
    loss_history.append(loss.item())

fig6, ax6 = plt.subplots(figsize=(6, 4))
vi.plot_loss_curve(loss_history, ax=ax6, title='CD training loss (60 epochs)')
plt.show()

## Test 7 — `plot_methods_grid`

并排对比几个不同容量的**未训练**网络（复用 Test 1 的思路，但用网格布局一次性展示多个）——检查子图布局、标题、以及多余子图是否被正确隐藏（3 张图放进 3 列网格，不应该有空白第四格残留坐标轴）。

In [ ]:
torch.manual_seed(0)
nets = {
    'hidden_dim=4, layers=1': EnergyNet(hidden_dim=4, n_hidden_layers=1),
    'hidden_dim=32, layers=2': EnergyNet(hidden_dim=32, n_hidden_layers=2),
    'hidden_dim=128, layers=3': EnergyNet(hidden_dim=128, n_hidden_layers=3),
}

fig7, axes7 = vi.plot_methods_grid(nets, data, ncols=3)
plt.show()

## ✅ 检查清单

跑完以上所有 cell 后，确认：

- [ ] Test 0：能看到 figure 工具栏（说明 `%matplotlib widget` 后端已生效，而不是退回 `inline`）
- [ ] Test 1：拖动滑块时同一张图原地变化，没有新图往下堆积
- [ ] Test 2：轨迹线条颜色由浅到深，末端有星号标记，叠加在正确的能量等高线上
- [ ] Test 3：拖动 step 滑块，轨迹逐步“生长”，当前点高亮圆点跟着移动
- [ ] Test 4：箭头场方向大致指向数据密集区域
- [ ] Test 5：拖动 noise_std 滑块，红色点云原地变化
- [ ] Test 6：loss 曲线整体下降
- [ ] Test 7：三个子图正确显示各自标题，没有多余的空白坐标轴

如果某一项在浏览器里表现为“图片不断往下堆积、不是原地更新”，最常见原因是 `ipympl` 未正确安装或 Jupyter 内核需要重启：

```bash
pip install ipympl
# 然后在 Jupyter 里 Kernel → Restart Kernel，再从头运行本 notebook
```